# Session 7: Data Aggregation & Summarization
**Python for Data Analysis Workshop | World Development Indicators**

---

## Learning Objectives

By the end of this session you will be able to:
- Use `groupby()` to split-apply-combine data
- Apply single and multiple aggregation functions with `.agg()`
- Use `.transform()` for within-group calculations
- Compute rolling windows and cumulative statistics
- Build ranked and comparative summaries
- Answer real analytical questions from the WDI dataset

---

## 1. Load the Data

In [ ]:
import pandas as pd
import numpy as np

try:
    df = pd.read_csv("WDI_clean.csv")
except FileNotFoundError:
    df = pd.read_csv("World_Dev_Indicators.csv")
    df["Region"] = df["Region"].str.strip()

print(f"Shape: {df.shape}")

# Convenient shorter column names for this session
df = df.rename(columns={
    "GDP (current US$)":                          "GDP",
    "GDP per capita (current US$)":               "GDPpc",
    "Life expectancy at birth, total (years)":    "LifeExp",
    "Population, total":                          "Population",
    "Suicide mortality rate (per 100,000 population)": "SuicideRate"
})
df.head(3)

## 2. The Split-Apply-Combine Pattern

`groupby()` follows three steps:
1. **Split** the data into groups based on one or more columns
2. **Apply** a function (mean, sum, count, custom) to each group
3. **Combine** the results back into a single DataFrame

Think of it like Excel's PivotTable or SQL's `GROUP BY`.

## 3. Basic `groupby()` with a Single Column

In [ ]:
# Mean GDP per capita by Income Group (across all years)
mean_gdppc = df.groupby("Income Group")["GDPpc"].mean().round(0)
print("Mean GDP per capita by Income Group:")
print(mean_gdppc.sort_values(ascending=False))

In [ ]:
# Count observations by Region
region_counts = df.groupby("Region")["Country"].nunique()
print("Unique countries per Region:")
print(region_counts.sort_values(ascending=False))

In [ ]:
# Multiple aggregations in one step
region_summary = df.groupby("Region").agg(
    Avg_GDPpc     = ("GDPpc",  "mean"),
    Median_LifeExp= ("LifeExp","median"),
    Country_Count = ("Country","nunique"),
    Total_Pop     = ("Population","sum")
).round(1)
print("Region summary:")
region_summary.sort_values("Avg_GDPpc", ascending=False)

## 4. Grouping by Multiple Columns

In [ ]:
# Average life expectancy by Region AND Year
le_by_region_year = df.groupby(["Region","Year"])["LifeExp"].mean().round(1)
print("Mean life expectancy by Region and Year (sample):")
le_by_region_year.head(14)

In [ ]:
# Reset index to work with it as a flat DataFrame
le_df = le_by_region_year.reset_index()
le_df.columns.name = None
print("\nFlat DataFrame:")
le_df.head(5)

## 5. The `.agg()` Function — Multiple Aggregations

In [ ]:
# Use .agg() with a list of functions
income_agg = df.groupby("Income Group")["GDPpc"].agg(["mean","median","std","min","max","count"])
income_agg = income_agg.round(0)
print("GDP per capita stats by Income Group:")
income_agg

In [ ]:
# Different aggregations for different columns
full_agg = df.groupby("Income Group").agg(
    GDPpc_Mean    = ("GDPpc",  "mean"),
    GDPpc_Median  = ("GDPpc",  "median"),
    LifeExp_Mean  = ("LifeExp","mean"),
    LifeExp_Max   = ("LifeExp","max"),
    Countries     = ("Country","nunique"),
    Rows          = ("Country","count")
).round(1)
print("Full aggregation by Income Group:")
full_agg

In [ ]:
# Custom aggregation function
def range_stat(s):
    """Return max - min (range)."""
    return s.max() - s.min()

custom_agg = df.groupby("Region")["LifeExp"].agg(
    Mean = "mean",
    Spread = range_stat
).round(1)
print("Life expectancy mean and range by Region:")
custom_agg.sort_values("Spread", ascending=False)

## 6. `.transform()` — Within-Group Calculations

`.transform()` applies a function per group but **returns a value for every row**,
preserving the original DataFrame shape. This is ideal for creating new columns
based on group statistics.

In [ ]:
# Add the regional average GDP per capita as a column (aligned to each row)
df["Region_Avg_GDPpc"] = df.groupby("Region")["GDPpc"].transform("mean")

# Now compute: how far is each country-year from its regional average?
df["GDPpc_vs_Region_Avg"] = df["GDPpc"] - df["Region_Avg_GDPpc"]

print("Sample — country vs regional average GDP per capita:")
df[["Country","Year","Region","GDPpc","Region_Avg_GDPpc","GDPpc_vs_Region_Avg"]]\
  .dropna().head(8)

In [ ]:
# Another example: each row gets its group's median life expectancy
df["Income_Median_LifeExp"] = df.groupby("Income Group")["LifeExp"].transform("median")

# Countries above their income group median
df["Above_Income_Median_LE"] = df["LifeExp"] > df["Income_Median_LifeExp"]
print("Countries above income group median life expectancy (2020):")
df[(df["Year"]==2020) & df["Above_Income_Median_LE"]]\
  .nlargest(8,"LifeExp")[["Country","Income Group","LifeExp","Income_Median_LifeExp"]]

## 7. Analytical Questions — Putting It Together

### Q1: Which region has seen the greatest improvement in life expectancy from 2006 to 2020?

In [ ]:
le_by_region = df.groupby(["Region","Year"])["LifeExp"].mean().reset_index()

le_2006 = le_by_region[le_by_region["Year"]==2006].set_index("Region")["LifeExp"]
le_2020 = le_by_region[le_by_region["Year"]==2020].set_index("Region")["LifeExp"]

le_change = (le_2020 - le_2006).round(2).sort_values(ascending=False)
le_change.name = "LE Change 2006-2020 (years)"
print("Life expectancy change by region (2006→2020):")
print(le_change)

### Q2: Top 5 countries by GDP per capita growth from 2010 to 2020

In [ ]:
gdp_pivot = df.pivot_table(index="Country", columns="Year", values="GDPpc", aggfunc="mean")
gdp_pivot = gdp_pivot[[2010, 2020]].dropna()
gdp_pivot["Growth %"] = ((gdp_pivot[2020] - gdp_pivot[2010]) / gdp_pivot[2010] * 100).round(1)
top_growers = gdp_pivot.sort_values("Growth %", ascending=False).head(10)
print("Top 10 countries by GDP per capita growth 2010→2020:")
top_growers

### Q3: Is there a relationship between income group and suicide rates?

In [ ]:
suicide_by_income = df.groupby("Income Group")["SuicideRate"].agg(
    Count  = "count",
    Mean   = "mean",
    Median = "median",
    StdDev = "std"
).round(2)
print("Suicide rate by Income Group:")
suicide_by_income

## 8. Rolling Windows and Cumulative Statistics

Useful for time-series analysis within each country.

In [ ]:
# 3-year rolling average GDP per capita for USA
usa = df[df["Country"] == "United States"].sort_values("Year").copy()
usa["GDPpc_3yr_avg"] = usa["GDPpc"].rolling(window=3, min_periods=1).mean().round(0)
print("USA GDP per capita with 3-year rolling average:")
usa[["Year","GDPpc","GDPpc_3yr_avg"]].tail(10)

In [ ]:
# Year-over-year change in GDP per capita per country
df_sorted = df.sort_values(["Country","Year"])
df["GDPpc_YoY_change"] = df_sorted.groupby("Country")["GDPpc"].diff()

print("Year-over-year GDP per capita change (sample):")
df[df["Country"]=="China"][["Year","GDPpc","GDPpc_YoY_change"]].dropna().tail(10)

In [ ]:
# Cumulative population by region over time
region_pop = df.groupby(["Region","Year"])["Population"].sum().reset_index()
region_pop = region_pop.sort_values(["Region","Year"])

# Show global total population per year
global_pop = region_pop.groupby("Year")["Population"].sum()
global_pop_df = global_pop.reset_index()
global_pop_df.columns = ["Year","Global Population"]
global_pop_df["Global Population (B)"] = (global_pop_df["Global Population"] / 1e9).round(2)
print("Global population by year:")
global_pop_df.tail(10)

## 9. Summary Cheat Sheet

| Task | Code |
|------|------|
| Group + mean | `df.groupby('col')['val'].mean()` |
| Group + multiple aggs | `df.groupby('col').agg(Name=('val','func'))` |
| Group by multiple cols | `df.groupby(['col1','col2'])` |
| Custom aggregation | `df.groupby('col').agg(my_func)` |
| Within-group calc | `df.groupby('col')['val'].transform('mean')` |
| Rolling average | `s.rolling(window=N).mean()` |
| Year-over-year change | `df.groupby('Country')['val'].diff()` |
| Flatten groupby result | `.reset_index()` |

---

## 10. Exercises

1. Calculate the **median** GDP per capita for each **Lending Type** in each **Year**. Which lending type consistently has the highest GDP per capita?
2. For each country, find the **year with the highest life expectancy**. Which countries peaked before 2020?
3. Use `.transform()` to add a column `Pct_of_Regional_Population` showing each country's share of its regional total population.
4. **Bonus:** Calculate the 5-year rolling average life expectancy for each country. Which country had the steepest rise in any 5-year window?

In [ ]:
# Your answers here:
# Exercise 1

# Exercise 2

# Exercise 3

# Exercise 4 (bonus)
